# 🔎 GIADA Gate D — dove costa l’ibrido?
Decomposizione compilata e appaiata di cinque canali appresi, sei formule residue, update dei gate e correnti analitiche. Diagnostica soltanto: il NO-GO del Gate D rimane invariato.


In [ ]:
from pathlib import Path
import base64,json,os,shutil,subprocess,sys,uuid
from IPython.display import display,Javascript
WORK=Path('/kaggle/working')/('giada_gate_d_forensic_'+uuid.uuid4().hex[:8]);WORK.mkdir()
REPO=WORK/'giada'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin',os.environ.get('GIADA_CODE_REVISION','codex/surrogate-validity-audit')],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
subprocess.run([sys.executable,'-c','import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name())'],check=True)
print({'revision':REVISION})


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts')/('giada_gate_d_forensic_'+REVISION[:7]+'_'+uuid.uuid4().hex[:8])
completed=subprocess.run([sys.executable,'-u',str(REPO/'scripts/run_gate_d_bottleneck_forensic.py'),'--output',str(OUTPUT)],check=False)
path=OUTPUT/('failure_report.json' if (OUTPUT/'failure_report.json').exists() else 'final_report.json')
report=json.loads(path.read_text()) if path.is_file() else {'valid':False,'error':f'worker interrupted: {completed.returncode}'}
display({k:report.get(k) for k in ('valid','diagnostic_only','gate_d_completed','task29_authorized','device_name','error')})
if report.get('rows'):
 display([{ 'family':r['family'],'median_ms':{n:round(v['median_ms'],4) for n,v in r['timing'].items()},'paired':r['paired_reduction_fractions'],'kernel_counts':{n:p.get('kernel_event_count') for n,p in r['profiles'].items()}} for r in report['rows']])
print({'output':str(OUTPUT),'exit':completed.returncode})


In [ ]:
archive=Path(shutil.make_archive(str(Path('/kaggle/working')/OUTPUT.name),'zip',OUTPUT))
if os.environ.get('GIADA_MANUAL_DOWNLOAD','0')=='1':
 payload=base64.b64encode(archive.read_bytes()).decode('ascii')
 display(Javascript(f"const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"))
print({'archive':str(archive),'size_mib':round(archive.stat().st_size/2**20,2)})
